# 1. Introducción a las vistas

En este notebook vamos a **crear y explorar vistas en Databricks**.

Primero crearemos **un ejemplo de cada tipo de vista**. Posteriormente, en otro notebook, analizaremos **las diferencias en su comportamiento en una nueva sesión**.

Para comenzar, creamos una **tabla de datos** que utilizaremos en esta demostración.

En este caso, creamos una tabla llamada **`smartphones`**, que contiene las siguientes columnas:
- `id`
- `name`
- `brand`
- `release_year`

A continuación, insertamos **algunos datos de ejemplo** en esta tabla para poder trabajar con ellos en los siguientes pasos.

In [0]:
USE CATALOG workspacedemofull02;

CREATE TABLE IF NOT EXISTS smartphones
(id INT, name STRING, brand STRING, year INT);

INSERT INTO smartphones
VALUES (1, 'iPhone 14', 'Apple', 2022),
      (2, 'iPhone 13', 'Apple', 2021),
      (3, 'iPhone 6', 'Apple', 2014),
      (4, 'iPad Air', 'Apple', 2013),
      (5, 'Galaxy S22', 'Samsung', 2022),
      (6, 'Galaxy Z Fold', 'Samsung', 2022),
      (7, 'Galaxy S9', 'Samsung', 2016),
      (8, '12 Pro', 'Xiaomi', 2022),
      (9, 'Redmi 11T Pro', 'Xiaomi', 2022),
      (10, 'Redmi Note 11', 'Xiaomi', 2021)

num_affected_rows,num_inserted_rows
10,10


## 1.1 Listado de tablas y vistas

También podemos utilizar el comando **`SHOW TABLES`** para visualizar el listado de **tablas y vistas disponibles**.

Al ejecutarlo, veremos las tablas y vistas que existen dentro del schema actual.

En nuestro caso, podemos observar que tenemos **una única tabla llamada `smartphones`** dentro de la base de datos **`default`**.

In [0]:
SHOW TABLES

database,tableName,isTemporary
default,smartphones,false
,_sqldf,true


## 1.2 Creación de una vista (stored view)

Vamos a crear ahora una **vista persistente (stored view)** que muestre únicamente los teléfonos de **Apple**.

Para ello utilizamos la sentencia **`CREATE VIEW`**, seguida del nombre de la vista.  
En nuestro caso, la llamaremos **`viewApplePhones`**.

A continuación, con la palabra clave **`AS`**, definimos la **consulta lógica** que representa la vista.

En este ejemplo, seleccionamos todos los registros de la tabla **`smartphones`** donde la columna **`brand`** es igual a **"Apple"**.

In [0]:
CREATE VIEW view_apple_phones
AS  SELECT * 
    FROM smartphones 
    WHERE brand = 'Apple';

## 1.3 Consultando la vista

Ahora podemos **consultar nuestra vista** utilizando una sentencia estándar de **`SELECT`**, igual que haríamos con una tabla.

Al ejecutar la consulta, obtendremos **todos los teléfonos de Apple** que están almacenados en la tabla **`smartphones`**.

Es importante recordar que **cada vez que se consulta una vista**, lo que realmente ocurre es que **se ejecuta la consulta lógica definida en la vista contra la tabla original**.

In [0]:
SELECT * FROM view_apple_phones;

## 1.4 Verificando la vista en el catálogo

Si ejecutamos de nuevo el comando **`SHOW TABLES`**, podremos observar que la vista ha sido **persistida en la base de datos `default`**.

Además, podemos comprobar que **no se trata de un objeto temporal**, sino de una **vista almacenada de forma permanente** dentro del catálogo.

In [0]:
SHOW TABLES;


## 1.5 Creación de una vista temporal

Ahora vamos a crear una **vista temporal**.

La sintaxis es muy similar a la anterior, pero en este caso añadimos la palabra clave **`TEMPORARY`** (o simplemente **`TEMP`**).

La consulta lógica de esta vista consistirá en **obtener la lista única de marcas (`brand`)** presentes en nuestra tabla **`smartphones`**.

In [0]:
CREATE TEMP VIEW temp_view_phones_brands
AS  SELECT DISTINCT brand
    FROM smartphones;

SELECT * FROM temp_view_phones_brands;

## 1.6 Verificación de la vista temporal

Si ejecutamos nuevamente el comando **`SHOW TABLES`**, podremos ver que la **vista temporal aparece en el listado**.

En la columna **`isTemporary`** se indica que esta vista es efectivamente **un objeto temporal**.

Al tratarse de una vista temporal, **no se persiste en ninguna base de datos**, sino que solo existe **durante la sesión actual**.

In [0]:
SHOW TABLES;


## 1.7 Creación de una vista global temporal

Por último, vamos a crear una **vista temporal global**.

Para ello, utilizamos la misma sintaxis que antes, pero añadiendo la palabra clave **`GLOBAL`** junto con `TEMPORARY`.

En nuestro caso, la vista se llamará **`global_temp_view_latest_phones`**.

La consulta lógica de esta vista consiste en **recuperar todos los smartphones cuya fecha de lanzamiento sea posterior a 2020**, y ordenar los resultados en **orden descendente**, de manera que **los teléfonos más recientes aparezcan primero**.

In [0]:
CREATE GLOBAL TEMP VIEW global_temp_view_latest_phones
AS SELECT * FROM smartphones
    WHERE year > 2020
    ORDER BY year DESC;

## 1.8 Consultando una vista temporal global

Para consultar una **vista temporal global** mediante una sentencia `SELECT`, es necesario utilizar el **calificador de base de datos `global_temp`**.

Esto se debe a que las vistas temporales globales se almacenan en una **base de datos especial llamada `global_temp`**, que es **temporal y está asociada al cluster**.

Por tanto, para acceder a este tipo de vistas, debemos referenciarlas como:

`global_temp.nombre_de_la_vista`

In [0]:
SELECT * FROM global_temp.global_temp_view_latest_phones;

## 1.9 Revisión final de tablas y vistas

Antes de continuar, vamos a revisar una vez más el listado de **tablas y vistas** utilizando `SHOW TABLES`.

En este listado **no aparecerá la vista temporal global**, ya que está asociada a la base de datos especial **`global_temp`** y no al schema actual.

Por tanto, las **vistas globales temporales no se muestran en este listado**, a menos que consultemos explícitamente la base de datos **`global_temp`**.

In [0]:
SHOW TABLES;

## 1.10 Listando vistas en la base de datos `global_temp`

Para mostrar las **tablas y vistas dentro de la base de datos `global_temp`**, utilizamos el comando:

`SHOW TABLES IN global_temp`

Al ejecutarlo, podremos ver la **vista temporal global `latest_phones`**, que está asociada a la base de datos **`global_temp`** y, por tanto, es también **un objeto temporal**.

Por otro lado, la **vista temporal (no global)** que creamos anteriormente, como por ejemplo `phones_brands`, **no está asociada a ninguna base de datos**, por lo que **aparece en el resultado de cualquier comando `SHOW TABLES` dentro de la sesión actual**.

In [0]:
SHOW TABLES IN global_temp;

In [0]:
SHOW TABLES

# 2. Comportamiento de tablas y vistas en nuevas sesiones

A continuación, vamos a demostrar cómo **las tablas y algunas vistas se mantienen entre distintas sesiones**, mientras que las **vistas temporales no**.

Para ello, abriremos un **nuevo notebook**, lo que implica trabajar en una **nueva sesión de Spark**, y comprobaremos cómo se comportan allí las vistas que hemos creado.

De esta forma podremos ver con claridad **qué objetos se persisten en el catálogo** y **cuáles solo existen durante la sesión en la que fueron creados**.

In [0]:
USE CATALOG workspacedemofull02;

## 2.1 Comprobando los objetos en una nueva sesión de Spark

En esta nueva sesión de Spark, comencemos ejecutando de nuevo el comando **`SHOW TABLES`**.

El resultado confirma varios comportamientos importantes:

- La tabla **`smartphones`** sigue existiendo, como era de esperar.
- La **vista persistente** que habíamos creado anteriormente, **`viewApplePhones`**, también sigue disponible en esta nueva sesión.
- Sin embargo, la **vista temporal** **ya no existe**, ya que este tipo de objeto solo vive durante la sesión en la que fue creado.

Esto demuestra que **las tablas y las vistas persistentes se mantienen entre sesiones**, mientras que **las vistas temporales no se conservan al abrir una nueva sesión de Spark**.

EJECUTAR EN NOTEBOOK NUEVO:

In [0]:
SHOW TABLES;

## 2.2 Comparación entre sesiones

Como podemos ver, al volver a la sesión anterior, **las tablas y vistas que habíamos creado siguen estando disponibles en ese entorno**.

Esto confirma que cada sesión mantiene su propio contexto, pero los objetos persistentes como **tablas y vistas almacenadas** siguen existiendo independientemente de la sesión.

Ahora volvamos al **nuevo notebook (nueva sesión de Spark)** para continuar analizando el comportamiento de estos objetos.

EJECUTAR EN NOTEBOOK PREVIO (ORIGINAL)

In [0]:
SHOW TABLES;

## 2.3 Limitaciones de las vistas temporales

Las **vistas temporales** no son accesibles fuera de la sesión en la que fueron creadas.

Por ejemplo, no estarán disponibles en los siguientes casos:
- Desde otro notebook (como acabamos de comprobar)
- Tras **desvincular y volver a vincular** un notebook a un cluster
- Después de **instalar un paquete Python**, lo que reinicia el intérprete
- Tras **reiniciar el cluster**

Sin embargo, surge una duda: ¿qué ocurre con las **vistas temporales globales**?

Para comprobarlo, podemos ejecutar el comando:

`SHOW TABLES IN global_temp`

EJECUTAR EN NOTEBOOK NUEVO:

In [0]:
SHOW TABLES IN global_temp;

## 2.4 Persistencia de las vistas temporales globales

Interesante, la **vista temporal global sigue existiendo** en esta nueva sesión.

Esto se debe a que, mientras el **cluster esté en ejecución**, la base de datos **`global_temp`** se mantiene activa.  
Por tanto, **cualquier notebook conectado al mismo cluster puede acceder a sus vistas temporales globales**.

Podemos comprobarlo fácilmente **consultando esta vista desde esta nueva sesión**, lo que confirma que este tipo de vistas **sí se comparten entre notebooks dentro del mismo cluster**.

EJECUTAR EN NOTEBOOK NUEVO:

In [0]:
SELECT * FROM global_temp.global_temp_view_latest_phones;

## 2.5 Consideraciones finales sobre vistas globales y limpieza

Como podemos ver, la **vista temporal global funciona correctamente en esta sesión**, ya que el cluster sigue en ejecución.

Sin embargo, es importante recordar que si **reiniciamos el cluster**, esta vista dejará de existir, ya que la base de datos **`global_temp`** es también **temporal**.

Por último, vamos a proceder a **eliminar la tabla y las vistas creadas** para dejar el entorno limpio.